
# SKIRTOR clumpy vs Silva+04 smooth-torus comparison (X-CIGALE Fig. 2)

Reproduces the SKIRTOR vs Fritz comparison from Yang et al. 2020
(X-CIGALE Fig. 2). Both libraries re-emit the same disc-absorbed
luminosity in the mid-IR; the mid-IR peak amplitude differs by
~0.5 dex because SKIRTOR's clumpy 3-D Stalevski+2016 RT redistributes
heating more efficiently into the bright NIR-MIR continuum than a
smooth-density torus. tengri does not ship Fritz+2006 directly; we
substitute Silva+04 (template-based smooth torus, the closest
contemporary analog) — the qualitative contrast (clumpy bright
MIR vs smooth fainter MIR) is preserved.

Both runs use the same Kubota-Done multicolor disc at log L_bol =
12.5 (in L_sun units), Type-1 viewing (cos_inc = 0.95), and are
peak-normalized so the SED *shape* difference is what stands out.

Reference: Yang et al. 2020, MNRAS 491, 740 (X-CIGALE) Fig. 2;
Stalevski et al. 2016, MNRAS 458, 2288 (SKIRTOR);
Silva, Maiolino & Granato 2004, MNRAS 355, 973.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_AA_PER_S = 2.998e18
# Negligible host SFH: total mass ~1e-10 Msun, completely subdominant
# to the AGN luminosity below. ``log_sfr`` was the legacy kwarg; current
# ``const`` SFH parametrizes by total mass over [start_gyr, end_gyr].
SFH = {"type": "const", "all_params": tengri.FIXED, "log_total_mass": -10.0}
DUST = {"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0}
ssp = tengri.load_ssp()


def predict_with_torus(torus_type: str) -> tuple[np.ndarray, np.ndarray]:
    model = tengri.SEDModel.build(
        ssp,
        sfh=SFH,
        dust=DUST,
        agn={
            "all_params": tengri.FIXED,
            "log_lbol": 12.5,
            "lum_ratio": 1.0,
            "cos_inc": 0.95,
            "disc": {"type": "multicolor", "all_params": tengri.FIXED},
            "torus": {"type": torus_type, "all_params": tengri.FIXED},
        },
        redshift=tengri.Fixed(0.0),
    )
    p = dict(model.spec.sample(jax.random.PRNGKey(0)))
    out = model.predict(p)
    wave = np.asarray(model.wavelengths)
    return wave, np.asarray(out.rest_sed())


wave_skir, l_skir = predict_with_torus("skirtor")
wave_silv, l_silv = predict_with_torus("silva04")
# SKIRTOR and silva04 return SEDs on different-length wavelength grids;
# regrid silva04 onto the SKIRTOR axis so the shared log_lam_nm lines up.
l_silv = np.interp(wave_skir, wave_silv, l_silv)
log_lam_nm = np.log10(wave_skir / 10.0)

# Match each curve at the disc UV (around 100 nm = 1000 Å) so the
# downstream torus-bump amplitude difference is what's on display.
ref = np.argmin(np.abs(log_lam_nm - 2.0))
l_skir_norm = l_skir / l_skir[ref]
l_silv_norm = l_silv / l_silv[ref]

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(log_lam_nm, np.log10(l_skir_norm), color="C0", lw=1.8, label="SKIRTOR")
ax.plot(log_lam_nm, np.log10(l_silv_norm), color="C1", lw=1.8, label="Silva+04 (Fritz analog)")
ax.set(
    xlim=(1.0, 6.0),
    ylim=(-3.0, 2.5),
    xlabel=r"$\log\lambda$  [nm, rest-frame]",
    ylabel=r"$\log L_\nu$  (normed)",
)
ax.text(0.85, 0.10, "Type 1", transform=ax.transAxes, va="bottom", fontsize=12)
ax.legend(loc="upper left", frameon=False, fontsize=10)
fig.tight_layout()
plt.savefig("plot_skirtor_vs_smooth_torus.png", dpi=150, bbox_inches="tight")